# Patrones de Arquitectura de Datos

## Objetivos

- Implementar **bus de datos** para integración
- Diseñar **microservicios de datos** con autonomía
- Aplicar **event-driven architecture** en detalle
- Patrones de **interoperabilidad** y **reutilización**
- Introducción a **Data as a Product**

## 1. Bus de Datos (Data Bus)

### Concepto

Un **bus de datos** es un canal centralizado para distribuir datos entre múltiples sistemas.

**Analogía:** Como un bus de transporte público donde:
- **Productores** publican datos (suben pasajeros)
- **Consumidores** se suscriben (bajan en su parada)
- **Bus** transporta sin conocer origen/destino

### Arquitectura

```
Productores              Data Bus (Kafka)          Consumidores
   │                          │                         │
   │ CRM (customers)          │                         │
   ├─────────────────────────▶│                         │
   │ ERP (orders)             │─────────────────────────▶│ Data Lake
   ├─────────────────────────▶│                         │
   │ Web (clicks)             │─────────────────────────▶│ Analytics
   ├─────────────────────────▶│                         │
   │ IoT (sensors)            │─────────────────────────▶│ ML Platform
   └─────────────────────────▶│                         │
```

### Ventajas

✅ **Desacoplamiento:** Productores y consumidores independientes
✅ **Escalabilidad:** Agregar fuentes/destinos sin modificar existentes
✅ **Replay:** Reprocesar datos históricos desde el bus
✅ **Multi-consumo:** Múltiples equipos consumen mismos datos

### Tecnologías Comunes

| Tecnología | Throughput | Latencia | Retención | Caso de Uso |
|------------|------------|----------|-----------|-------------|
| **Kafka** | Muy alto | Baja (ms) | Días/semanas | Real-time, high volume |
| **RabbitMQ** | Medio | Baja | Horas | Mensajería general |
| **AWS Kinesis** | Alto | Baja | Días | AWS-native streaming |
| **Azure Event Hub** | Alto | Baja | Días | Azure-native streaming |
| **Google Pub/Sub** | Alto | Baja | Días | GCP-native messaging |

In [ ]:
# Simulación de Data Bus con Python (concepto)
from collections import defaultdict
from datetime import datetime
import json

class DataBus:
    """Simulador simple de un bus de datos"""
    
    def __init__(self):
        self.topics = defaultdict(list)  # topic -> lista de mensajes
        self.subscribers = defaultdict(list)  # topic -> lista de consumidores
    
    def publish(self, topic, message, producer):
        """Publicar mensaje en un topic"""
        event = {
            'timestamp': datetime.now().isoformat(),
            'producer': producer,
            'topic': topic,
            'payload': message
        }
        self.topics[topic].append(event)
        print(f"📤 [{producer}] publicó en '{topic}': {message}")
        
        # Notificar a suscriptores
        for subscriber in self.subscribers[topic]:
            subscriber(event)
    
    def subscribe(self, topic, consumer_name, callback):
        """Suscribirse a un topic"""
        self.subscribers[topic].append(callback)
        print(f"📥 [{consumer_name}] se suscribió a '{topic}'")
    
    def get_history(self, topic):
        """Obtener historial de mensajes de un topic"""
        return self.topics.get(topic, [])

# Crear bus
bus = DataBus()

# Definir consumidores
def analytics_consumer(event):
    print(f"  🔍 [Analytics] procesó: {event['payload'].get('order_id')}")

def notification_consumer(event):
    email = event['payload'].get('customer_email')
    print(f"  📧 [Notifications] envió email a: {email}")

def inventory_consumer(event):
    product = event['payload'].get('product_id')
    print(f"  📦 [Inventory] actualizó stock del producto: {product}")

# Suscribir consumidores
bus.subscribe('orders.created', 'Analytics', analytics_consumer)
bus.subscribe('orders.created', 'Notifications', notification_consumer)
bus.subscribe('orders.created', 'Inventory', inventory_consumer)

print("\n" + "="*60)
print("SIMULACIÓN DE EVENTOS")
print("="*60 + "\n")

# Publicar eventos
bus.publish('orders.created', {
    'order_id': 'ORD-001',
    'customer_email': 'juan@example.com',
    'product_id': 'PROD-123',
    'amount': 150.00
}, producer='OrderService')

print()

bus.publish('orders.created', {
    'order_id': 'ORD-002',
    'customer_email': 'maria@example.com',
    'product_id': 'PROD-456',
    'amount': 250.00
}, producer='OrderService')

print("\n" + "="*60)
print(f"📊 Total eventos en 'orders.created': {len(bus.get_history('orders.created'))}")

## 2. Microservicios de Datos

### Definición

**Microservicios de datos** son componentes autónomos que:
- Gestionan un dominio específico de datos
- Tienen su propio storage y lógica
- Se comunican via APIs o eventos
- Pueden desplegarse independientemente

### Principios Clave

1. **Bounded Context:** Cada servicio tiene límites claros
2. **Ownership:** Un equipo es dueño de principio a fin
3. **API-first:** Contratos bien definidos
4. **Autonomía:** Independiente en datos y despliegue

### Ejemplo: Plataforma E-commerce

```
┌───────────────────┐   ┌───────────────────┐   ┌───────────────────┐
│  Customer Data    │   │  Product Data     │   │  Order Data       │
│  Service          │   │  Service          │   │  Service          │
│                   │   │                   │   │                   │
│  • profiles       │   │  • catalog        │   │  • orders         │
│  • preferences    │   │  • inventory      │   │  • shipments      │
│  • segments       │   │  • pricing        │   │  • invoices       │
│                   │   │                   │   │                   │
│  [Customer DB]    │   │  [Product DB]     │   │  [Order DB]       │
└───────────────────┘   └───────────────────┘   └───────────────────┘
         │                       │                       │
         └───────────────────────┴───────────────────────┘
                              │
                    ┌─────────▼─────────┐
                    │   API Gateway     │
                    │  (GraphQL/REST)   │
                    └───────────────────┘
```

### Patrones de Comunicación

#### A. Síncrona (APIs REST/GraphQL)
```
Order Service  ──GET /customers/123──▶  Customer Service
               ◀──────────────────────  { name, email }
```

**Ventajas:** Simple, respuesta inmediata
**Desventajas:** Acoplamiento, latencia acumulada

#### B. Asíncrona (Eventos)
```
Order Service  ──order.created──▶  Event Bus  ──▶  Customer Service
                                                 ──▶  Inventory Service
```

**Ventajas:** Desacoplamiento, no bloquea
**Desventajas:** Eventual consistency, debugging complejo

In [ ]:
# Simulación de microservicios de datos
import pandas as pd
import random

class CustomerDataService:
    """Microservicio: Datos de clientes"""
    
    def __init__(self):
        # BD propia del servicio
        self.customers = pd.DataFrame({
            'customer_id': [101, 102, 103],
            'name': ['Juan Pérez', 'María García', 'Pedro López'],
            'email': ['juan@example.com', 'maria@example.com', 'pedro@example.com'],
            'segment': ['Premium', 'Standard', 'Premium']
        })
    
    def get_customer(self, customer_id):
        """API: Obtener información de cliente"""
        result = self.customers[self.customers['customer_id'] == customer_id]
        if len(result) == 0:
            return None
        return result.iloc[0].to_dict()

class ProductDataService:
    """Microservicio: Datos de productos"""
    
    def __init__(self):
        self.products = pd.DataFrame({
            'product_id': ['P001', 'P002', 'P003'],
            'name': ['Laptop', 'Mouse', 'Teclado'],
            'price': [1200, 25, 80],
            'stock': [15, 100, 50]
        })
    
    def get_product(self, product_id):
        """API: Obtener información de producto"""
        result = self.products[self.products['product_id'] == product_id]
        if len(result) == 0:
            return None
        return result.iloc[0].to_dict()
    
    def check_stock(self, product_id, quantity):
        """API: Verificar disponibilidad"""
        product = self.get_product(product_id)
        if product is None:
            return False
        return product['stock'] >= quantity

class OrderDataService:
    """Microservicio: Datos de pedidos"""
    
    def __init__(self, customer_service, product_service):
        self.customer_service = customer_service
        self.product_service = product_service
        self.orders = []
    
    def create_order(self, customer_id, product_id, quantity):
        """API: Crear pedido (orquesta llamadas a otros servicios)"""
        print(f"\n📝 Creando orden...")
        
        # 1. Validar cliente
        customer = self.customer_service.get_customer(customer_id)
        if customer is None:
            return {'success': False, 'error': 'Cliente no existe'}
        print(f"  ✅ Cliente validado: {customer['name']}")
        
        # 2. Validar producto y stock
        product = self.product_service.get_product(product_id)
        if product is None:
            return {'success': False, 'error': 'Producto no existe'}
        
        if not self.product_service.check_stock(product_id, quantity):
            return {'success': False, 'error': 'Stock insuficiente'}
        print(f"  ✅ Producto validado: {product['name']} (stock: {product['stock']})")
        
        # 3. Crear orden
        order_id = f"ORD-{random.randint(1000, 9999)}"
        order = {
            'order_id': order_id,
            'customer_id': customer_id,
            'customer_name': customer['name'],
            'product_id': product_id,
            'product_name': product['name'],
            'quantity': quantity,
            'total': product['price'] * quantity
        }
        self.orders.append(order)
        
        print(f"  ✅ Orden creada: {order_id} (Total: ${order['total']:.2f})")
        return {'success': True, 'order': order}

# Inicializar microservicios
customer_svc = CustomerDataService()
product_svc = ProductDataService()
order_svc = OrderDataService(customer_svc, product_svc)

print("=" * 60)
print("SIMULACIÓN DE MICROSERVICIOS DE DATOS")
print("=" * 60)

# Caso 1: Orden exitosa
result1 = order_svc.create_order(customer_id=101, product_id='P001', quantity=2)

# Caso 2: Cliente no existe
result2 = order_svc.create_order(customer_id=999, product_id='P001', quantity=1)
print(f"\n❌ Error: {result2['error']}")

# Caso 3: Stock insuficiente
result3 = order_svc.create_order(customer_id=102, product_id='P002', quantity=500)
print(f"\n❌ Error: {result3['error']}")

print(f"\n📊 Total órdenes creadas: {len(order_svc.orders)}")

## 3. Event-Driven Architecture Detallada

### Patrones de Procesamiento

#### A. Event Sourcing

**Concepto:** El estado se reconstruye desde la secuencia de eventos.

```
Eventos:                      Estado Actual:
1. AccountCreated(id=123)  →  balance: $0
2. Deposited($100)         →  balance: $100
3. Withdrew($30)           →  balance: $70
4. Deposited($50)          →  balance: $120
```

**Ventajas:**
- Auditoría completa (historia inmutable)
- Replay para debugging
- Temporal queries (¿cuál era el estado ayer?)

#### B. CQRS (Command Query Responsibility Segregation)

**Concepto:** Separar escrituras (commands) de lecturas (queries).

```
Write Model (Commands)         Read Model (Queries)
       │                              │
   [Commands]                    [Queries]
       │                              │
  Write DB ────Events────▶ Read DB (optimizado)
```

**Ventajas:**
- Escalar lecturas/escrituras independientemente
- Optimizar modelos de lectura (desnormalizado)

### Patrones de Garantías

| Garantía | Descripción | Trade-off |
|----------|-------------|------------|
| **At-most-once** | Mensaje procesado 0 o 1 vez | Puede perderse |
| **At-least-once** | Mensaje procesado ≥1 vez | Puede duplicarse |
| **Exactly-once** | Mensaje procesado exactamente 1 vez | Complejo, costoso |

## 4. Interoperabilidad y Reutilización

### Data Contracts

**Definición:** Acuerdo formal entre productor y consumidor sobre estructura, calidad y SLAs.

```yaml
data_contract:
  dataset: gold.customer_360
  owner: customer-analytics-team
  version: 2.1.0
  
  schema:
    - name: customer_id
      type: integer
      required: true
      unique: true
    - name: email
      type: string
      required: true
      format: email
    - name: segment
      type: string
      enum: [Premium, Standard, Basic]
  
  sla:
    freshness: 15 minutes
    availability: 99.9%
    completeness: 95%
  
  consumers:
    - marketing-team
    - sales-team
    - support-team
```

### API de Datos

**REST API:**
```
GET /api/v1/customers/123
GET /api/v1/customers?segment=Premium&limit=100
```

**GraphQL:**
```graphql
query {
  customer(id: 123) {
    name
    email
    orders(last: 10) {
      orderId
      total
    }
  }
}
```

## 5. Data as a Product

### Principios

Tratar datos como **productos** con:

1. **Ownership claro:** Un equipo es responsable de principio a fin
2. **Discoverability:** Fácil de encontrar en catálogo
3. **Quality:** SLAs de frescura, completitud, precisión
4. **Security:** Controles de acceso y enmascaramiento
5. **Usability:** Documentación, ejemplos, soporte

### Ejemplo: Customer 360 Data Product

```
📦 Data Product: Customer 360
├── 📄 README.md                 # Documentación
├── 📊 Dataset: gold.customer_360
├── 🔐 Access Policy: RBAC
├── 📝 Data Contract: v2.1.0
├── 🔍 Lineage: bronze → silver → gold
├── 📈 SLAs:
│   ├── Freshness: < 15 min
│   ├── Availability: 99.9%
│   └── Completeness: > 95%
├── 📚 Examples:
│   ├── SQL queries
│   └── Python snippets
└── 👥 Support: customer-analytics@company.com
```

### Métricas de Producto

- **Adoption:** # usuarios activos, # queries/día
- **Quality:** % registros válidos, % nulls en campos críticos
- **Performance:** P95 query time, throughput
- **Satisfaction:** NPS de usuarios, tickets de soporte

## 6. Ejercicio Integrador

**Escenario:** Plataforma de streaming de video

**Dominios de datos:**
1. **Users:** Perfiles, preferencias, suscripciones
2. **Content:** Películas, series, metadata
3. **Viewing:** Historial de reproducción, progress
4. **Recommendations:** ML model predictions

**Requisitos:**
- 100M usuarios activos
- 10K eventos/segundo (play, pause, etc)
- Recomendaciones en tiempo real (<500ms)

**Tareas:**

### A. Diseño de Microservicios (30%)
- Definir 4 microservicios de datos
- Especificar APIs de cada uno (3 endpoints)
- Diagramar comunicación entre servicios

### B. Event-Driven Architecture (30%)
- Identificar 5 eventos de negocio
- Diseñar topics de Kafka con particionamiento
- Definir garantías (at-least-once vs exactly-once)

### C. Data as a Product (20%)
- Diseñar data contract para "Viewing History"
- Definir SLAs (freshness, availability, quality)
- Listar consumidores y sus use cases

### D. Interoperabilidad (20%)
- Proponer cómo "Recommendations" consume datos de otros servicios
- REST síncrono vs eventos asíncronos (justificar)
- Diseñar estrategia de versionado de APIs

## 7. Resumen

### ✅ Patrones Implementados

**Bus de Datos:**
- Canal centralizado para distribución
- Desacoplamiento y multi-consumo
- Tecnologías: Kafka, Kinesis, Event Hub

**Microservicios de Datos:**
- Autonomía por dominio
- Comunicación síncrona (APIs) y asíncrona (eventos)
- Ownership de principio a fin

**Event-Driven:**
- Event Sourcing para auditoría
- CQRS para separar lecturas/escrituras
- Garantías: at-most-once, at-least-once, exactly-once

**Data as a Product:**
- Ownership, discoverability, quality, security, usability
- Data contracts con SLAs
- Métricas de adopción y satisfacción

### 🎯 Próximo Nivel (Senior)

- Arquitecturas de referencia completas
- Comparativas tradicionales vs modernas
- Data Mesh y domain-oriented decentralization
- Trade-offs arquitectónicos a escala empresarial

### 📚 Recursos

- "Building Microservices" - Sam Newman
- "Data Mesh" - Zhamak Dehghani
- Martin Fowler: Event Sourcing, CQRS
- Confluent Kafka documentation